# Bộ kết quả cuối cùng — Data Mining

Phạm vi đã đóng: tuning COMPLETE, E1 16/16, E2 12/12; E3–E5 DEFERRED; TIGER/CRAMER THEORY_ONLY; E7 recorded analysis COMPLETE, dedicated profiling DEFERRED; E8 COMPLETE. Không training/tuning/inference/profiling trong notebook. Public mode đọc tables đã versioned, chạy được từ clean clone.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/course_final.py").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.course_final import analyze, ARCHIVE_NAME, ARCHIVE_SHA256
from src.analysis.master_session1 import BUNDLE_HASH
TABLES = ROOT / "results/tables/course_final"
FULL_AUDIT = False  # True requires three private source ZIPs and the matching dataset.
REBUILD_PUBLICATION = False
if FULL_AUDIT:
    archive = Path(os.environ.get("SEQREC_FINAL_ARCHIVE", str(ROOT / "artifacts/e2_final" / ARCHIVE_NAME)))
    bundle = Path(os.environ.get("SEQREC_BUNDLE", str(ROOT / "data/processed/beauty" / BUNDLE_HASH)))
    analyze(archive, ROOT, bundle, 10000)
if REBUILD_PUBLICATION:
    from gen_fig_course_final import generate
    from write_course_final_report import write_report
    generate(ROOT)
    write_report(ROOT)
audit = json.loads((TABLES / "artifact_audit.json").read_text())
assert audit["state"] == "VERIFIED" and audit["archive_sha256"] == ARCHIVE_SHA256
assert audit["runs_verified"] == 28 and audit["prior_changed_files"] == 0
assert audit["training_closed"] and audit["remaining_requested_training_runs"] == 0
print("Raw re-audit" if FULL_AUDIT else "Public audited-table mode, not a fresh raw audit")
print("Source:", ARCHIVE_SHA256)


In [ ]:
display(pd.read_csv(TABLES / "course_status.csv"))
display(pd.read_csv(TABLES / "scientific_findings.csv"))


## Validation tuning và official test phải tách biệt

Ba trial/mô hình seed 7 là bounded search, không phải exhaustive optimum. Tuning selection chỉ dựa validation; E1/E2 là held-out test, không dùng chọn lại hyperparameters hoặc scope. MostPopular n=1 và SD undefined.


In [ ]:
tuning = pd.read_csv(TABLES / "tuning_selected.csv")
assert (tuning.evaluation_split == "validation").all()
display(tuning)
summary = pd.read_csv(TABLES / "metric_summary.csv")
assert (summary.evaluation_split == "test").all()
display(summary)


In [ ]:
display(Image(filename=str(ROOT / "results/figures/course_final/e1_test_metrics.png")))
display(Image(filename=str(ROOT / "results/figures/course_final/e2_test_metrics.png")))


## E7 và E8 từ artifacts

Không đo mới GPU/latency. Timing scope và giới hạn phải được giữ khi viết báo cáo. E8 chỉ đọc saved E1 predictions đã được chứng minh không thay đổi.


In [ ]:
display(pd.read_csv(TABLES / "e2_efficiency_summary.csv"))
display(pd.read_csv(TABLES / "e8_error_groups.csv"))
display(pd.read_csv(TABLES / "asset_catalog.csv"))


In [ ]:
display(Image(filename=str(ROOT / "results/figures/course_final/recorded_efficiency.png")))
display(Image(filename=str(ROOT / "results/figures/course_final/e8_error_groups.png")))
